In [ ]:
# scripts/visualize.py
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120

DATA_PATH = "data/dataset.csv"   # <- change to your dataset path
SAVE_DIR = "reports/figures"

def ensure_dir(d):
    os.makedirs(d, exist_ok=True)

def load_data(path=DATA_PATH):
    return pd.read_csv(path)

def save_fig(fig, name, save_dir=SAVE_DIR, dpi=150):
    ensure_dir(save_dir)
    path = os.path.join(save_dir, name)
    fig.savefig(path, bbox_inches="tight", dpi=dpi)
    plt.close(fig)

def plot_histograms(df, numeric_cols=None, bins=30):
    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include="number").columns.tolist()
    for col in numeric_cols:
        fig, ax = plt.subplots(figsize=(6,4))
        sns.histplot(df[col].dropna(), bins=bins, kde=True, ax=ax, color="C0")
        ax.set_title(f"Histogram of {col}")
        save_fig(fig, f"hist_{col}.png")

def plot_boxplots(df, numeric_cols=None, by=None):
    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include="number").columns.tolist()
    if by is None:
        # single boxplot per numeric column
        for col in numeric_cols:
            fig, ax = plt.subplots(figsize=(6,4))
            sns.boxplot(x=df[col].dropna(), ax=ax, color="C1")
            ax.set_title(f"Boxplot of {col}")
            save_fig(fig, f"box_{col}.png")
    else:
        # boxplot of each numeric column grouped by categorical 'by'
        for col in numeric_cols:
            fig, ax = plt.subplots(figsize=(8,5))
            sns.boxplot(x=by, y=col, data=df, ax=ax)
            ax.set_title(f"Boxplot of {col} by {by}")
            save_fig(fig, f"box_{col}_by_{by}.png")

def plot_pie_charts(df, cat_cols=None, top_n=10):
    if cat_cols is None:
        cat_cols = df.select_dtypes(include="object").columns.tolist()
    for col in cat_cols:
        counts = df[col].value_counts(dropna=False)
        if len(counts) > top_n:
            counts = counts.iloc[:top_n]
            counts["Other"] = df[col].value_counts().iloc[top_n:].sum()
        fig, ax = plt.subplots(figsize=(6,6))
        ax.pie(counts.values, labels=counts.index, autopct="%1.1f%%", startangle=90)
        ax.set_title(f"Pie chart of {col}")
        save_fig(fig, f"pie_{col}.png")

def plot_countplots(df, cat_cols=None):
    if cat_cols is None:
        cat_cols = df.select_dtypes(include=["object","category"]).columns.tolist()
    for col in cat_cols:
        fig, ax = plt.subplots(figsize=(8,4))
        sns.countplot(data=df, x=col, order=df[col].value_counts().index, ax=ax)
        ax.set_title(f"Count plot of {col}")
        plt.xticks(rotation=45, ha="right")
        save_fig(fig, f"count_{col}.png")

def plot_corr_heatmap(df, numeric_cols=None):
    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include="number").columns.tolist()
    corr = df[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
    ax.set_title("Correlation heatmap")
    save_fig(fig, "corr_heatmap.png")

def plot_pairplot(df, numeric_cols=None, hue=None, sample_frac=0.2, max_vars=6):
    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include="number").columns.tolist()
    # limit number of variables for readability
    numeric_cols = numeric_cols[:max_vars]
    sample = df[numeric_cols + ([hue] if hue else [])].dropna().sample(frac=sample_frac, random_state=1)
    g = sns.pairplot(sample, hue=hue, diag_kind="kde", corner=True)
    ensure_dir(SAVE_DIR)
    g.savefig(os.path.join(SAVE_DIR, "pairplot.png"))
    plt.close()

if __name__ == "__main__":
    df = load_data()
    # Example usage - adapt columns as needed
    plot_histograms(df, numeric_cols=["duration","src_bytes","dst_bytes"])
    plot_boxplots(df, numeric_cols=["duration","src_bytes"], by="label")  # replace 'label' with your class column
    plot_pie_charts(df, cat_cols=["protocol_type","service"])
    plot_countplots(df, cat_cols=["label"])
    plot_corr_heatmap(df)
    plot_pairplot(df, numeric_cols=["duration","src_bytes","dst_bytes","wrong_fragment","urgent"], hue="label")
